<a href="https://colab.research.google.com/github/e22098/Statistics-Learning-e22098/blob/main/Copy_of_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# 1. Setup Simulation Parameters
np.random.seed(42)
n_items = 20
theta_true = 0.75

# Initialize discrete grid for theta
theta_min, theta_max, grid_points = -4, 4, 400
theta_grid = np.linspace(theta_min, theta_max, grid_points)
d_theta = theta_grid[1] - theta_grid[0]

# Initialize Prior N(0,1)
posterior_grid = norm.pdf(theta_grid, loc=0, scale=1)

# Arrays to store the running estimates at each step
map_estimates = [theta_grid[np.argmax(posterior_grid)]]
mean_estimates = [np.sum(theta_grid * posterior_grid) * d_theta]
steps = [0]

# 2. Simulate the Sequential Assessment Loop
for k in range(1, n_items + 1):
    # Draw random item parameters
    a_k = np.random.uniform(0.5, 2.0)
    b_k = np.random.normal(0, 1)

    # Calculate user's true probability of success and simulate response
    p_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.rand() < p_true else 0

    # Calculate likelihood array across the entire theta grid
    p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))

    # Bayesian Update: multiply prior by likelihood
    unnormalized_posterior = likelihood * posterior_grid

    # Normalize using Riemann sum approximation
    marginal_likelihood = np.sum(unnormalized_posterior) * d_theta
    posterior_grid = unnormalized_posterior / marginal_likelihood

    # Extract MAP (mode) and EAP (mean) estimates
    current_map = theta_grid[np.argmax(posterior_grid)]
    current_mean = np.sum(theta_grid * posterior_grid) * d_theta

    map_estimates.append(current_map)
    mean_estimates.append(current_mean)
    steps.append(k)

# 3. Plotting the Convergence
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=mean_estimates, mode='lines+markers',
    name='Posterior Mean (EAP)', line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=steps, y=map_estimates, mode='lines+markers',
    name='MAP Estimate', line=dict(color='orange')
))

# Add reference line for the user's true hidden ability
fig.add_hline(y=theta_true, line_dash="dash", line_color="green",
              annotation_text=f"True Theta ({theta_true})")

fig.update_layout(
    title='Convergence of Latent Ability Estimators (20 Items)',
    xaxis_title='Item Step (k)',
    yaxis_title='Estimated Latent Ability (θ)',
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    template='plotly_white'
)
fig.show()

### 2. Sequential Likelihood Contribution

Since the response $Y_i$ to each item is binary, it follows a Bernoulli distribution conditional on $\theta$. The likelihood contribution of a single new observation $y_k \in \{0,1\}$ at step $k$ is:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Assuming local independence (the responses to different items are independent given the underlying ability $\theta$), the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is simply the product of the individual likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k \left[ \frac{1}{1+e^{-a_i(\theta-b_i)}} \right]^{y_i} \left[ 1 - \frac{1}{1+e^{-a_i(\theta-b_i)}} \right]^{1-y_i}$$

---

### 3. Mathematical Formulation of the Running Update

Bayesian sequential updating uses the posterior distribution from step $k-1$ as the prior distribution for step $k$. Up to a proportionality constant, the recursive relationship is defined by Bayes' theorem as:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

To obtain the exact posterior density, we divide by the marginal likelihood (the normalizing constant) found by integrating over the entire parameter space of $\theta$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{-\infty}^{\infty} L(y_k \mid u) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(u \mid \mathbf{y}^{(k-1)}) \, du}$$

---

### 4. Dynamic Shifting

If the user answers a highly difficult item correctly, we have $y_k = 1$ and a large positive $b_k$.

In this scenario, the likelihood contribution $L(y_k=1 \mid \theta) = p_k(\theta)$ is very close to $0$ for all ability levels below $b_k$ and only climbs toward $1$ for high values of $\theta$. When we multiply the running prior density $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ by this strict likelihood function, the probabilities for lower $\theta$ values are heavily penalized (driven close to zero). Consequently, the remaining probability mass must shift significantly to the right after normalization. This mathematically forces the peak of the new posterior density to shift to a higher estimated ability level.

---

### 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ acts as a slope multiplier in the logistic exponent, dictating the steepness of the likelihood function around the threshold $b_k$.

*   **When $a_k$ is very large:** The likelihood function is a steep step-function. It sharply penalizes $\theta$ values on the "wrong" side of the difficulty threshold. Multiplying the prior by this steep function aggressively trims the tails of the distribution. This dramatically reduces the variance of the posterior, increasing its "sharpness" and meaning the platform gained a high degree of certainty about the user's ability bounds.
*   **When $a_k$ is very small:** The likelihood function is very flat across the $\theta$ continuum. A flat likelihood multiplier barely alters the shape of the prior distribution. The updated posterior will look nearly identical to the prior, meaning the test variance remains largely unchanged and the item provided almost no useful information to update the system's certainty.

---

### 6. Numerical Implementation of a Running Grid

Since the integral in the denominator of the Bayesian update does not have a closed-form analytical solution for the 2PL model, we approximate it computationally using a discrete grid.

1.  **Grid Initialization:** Define a discrete, evenly spaced array of $M$ values for $\theta$, such as $\theta_j$ from $-4$ to $4$ with a step size $\Delta\theta$. Initialize the prior array $P^{(0)}(\theta_j) = \frac{1}{\sqrt{2\pi}} \exp(-\frac{\theta_j^2}{2})$.
2.  **Sequential Likelihood Computation:** For each new item $k$ with response $y_k$, compute the likelihood array for every grid point: $L_j = [p_k(\theta_j)]^{y_k} [1 - p_k(\theta_j)]^{1 - y_k}$.
3.  **Element-wise Multiplication:** Compute the unnormalized posterior array by multiplying the previous posterior array by the new likelihood array: $U_j = L_j \times P^{(k-1)}(\theta_j)$.
4.  **Numerical Normalization:** Approximate the continuous integral in the denominator using a Riemann sum: $S = \sum_{j=1}^M U_j \Delta\theta$. Finally, divide the unnormalized array by $S$ to yield the updated posterior density: $P^{(k)}(\theta_j) = \frac{U_j}{S}$.

---

### 7. Evaluating Convergence over the Timeline

```python
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# 1. Setup Simulation Parameters
np.random.seed(42)
n_items = 20
theta_true = 0.75

# Initialize discrete grid for theta
theta_min, theta_max, grid_points = -4, 4, 400
theta_grid = np.linspace(theta_min, theta_max, grid_points)
d_theta = theta_grid[1] - theta_grid[0]

# Initialize Prior N(0,1)
posterior_grid = norm.pdf(theta_grid, loc=0, scale=1)

# Arrays to store the running estimates at each step
map_estimates = [theta_grid[np.argmax(posterior_grid)]]
mean_estimates = [np.sum(theta_grid * posterior_grid) * d_theta]
steps = [0]

# 2. Simulate the Sequential Assessment Loop
for k in range(1, n_items + 1):
    # Draw random item parameters
        a_k = np.random.uniform(0.5, 2.0)
            b_k = np.random.normal(0, 1)
                
                    # Calculate user's true probability of success and simulate response
                        p_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
                            y_k = 1 if np.random.rand() < p_true else 0
                                
                                    # Calculate likelihood array across the entire theta grid
                                        p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
                                            likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))
                                                
                                                    # Bayesian Update: multiply prior by likelihood
                                                        unnormalized_posterior = likelihood * posterior_grid
                                                            
                                                                # Normalize using Riemann sum approximation
                                                                    marginal_likelihood = np.sum(unnormalized_posterior) * d_theta
                                                                        posterior_grid = unnormalized_posterior / marginal_likelihood
                                                                            
                                                                                # Extract MAP (mode) and EAP (mean) estimates
                                                                                    current_map = theta_grid[np.argmax(posterior_grid)]
                                                                                        current_mean = np.sum(theta_grid * posterior_grid) * d_theta
                                                                                            
                                                                                                map_estimates.append(current_map)
                                                                                                    mean_estimates.append(current_mean)
                                                                                                        steps.append(k)

                                                                                                        # 3. Plotting the Convergence
                                                                                                        fig = go.Figure()

                                                                                                        fig.add_trace(go.Scatter(
                                                                                                            x=steps, y=mean_estimates, mode='lines+markers',
                                                                                                                name='Posterior Mean (EAP)', line=dict(color='blue')
                                                                                                                ))

                                                                                                                fig.add_trace(go.Scatter(
                                                                                                                    x=steps, y=map_estimates, mode='lines+markers',
                                                                                                                        name='MAP Estimate', line=dict(color='orange')
                                                                                                                        ))

                                                                                                                        # Add reference line for the user's true hidden ability
                                                                                                                        fig.add_hline(y=theta_true, line_dash="dash", line_color="green",
                                                                                                                                      annotation_text=f"True Theta ({theta_true})")

                                                                                                                                      fig.update_layout(
                                                                                                                                          title='Convergence of Latent Ability Estimators (20 Items)',
                                                                                                                                              xaxis_title='Item Step (k)',
                                                                                                                                                  yaxis_title='Estimated Latent Ability (θ)',
                                                                                                                                                      xaxis=dict(tickmode='linear', tick0=0, dtick=2),
                                                                                                                                                          template='plotly_white'
                                                                                                                                                          )
                                                                                                                                                          fig.show()
                                                                                                                                                          

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

In [ ]:
# ### 1. Structural Probability and Properties
#
# The Beta distribution is highly flexible, and its shape is determined by the balance between its two shape parameters, $\alpha$ and $\beta$.
#
# *   **Uninformative state $(\alpha=1, \beta=1)$:** The distribution is a uniform flat line. Every value of $\theta \in [0,1]$ is equally probable.
# *   **Right-skewed state $(\alpha=2, \beta=8)$:** The center of mass shifts toward the left (closer to 0). This represents a belief that the event (a click) is rare.
# *   **Left-skewed state $(\alpha=8, \beta=2)$:** The center of mass shifts toward the right (closer to 1). This represents a belief that the event is highly likely.
#
# ```python
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_vals = np.linspace(0, 1, 500)

fig = go.Figure()

# Alpha=1, Beta=1 (Uniform/Uninformative)
fig.add_trace(go.Scatter(
    x=theta_vals, y=beta.pdf(theta_vals, 1, 1),
    mode='lines', name='α=1, β=1 (Uniform)',
    line=dict(color='blue')
))

# Alpha=2, Beta=8 (Right-skewed, peak near 0)
fig.add_trace(go.Scatter(
    x=theta_vals, y=beta.pdf(theta_vals, 2, 8),
    mode='lines', name='α=2, β=8 (Right-skewed)',
    line=dict(color='green')
))

# Alpha=8, Beta=2 (Left-skewed, peak near 1)
fig.add_trace(go.Scatter(
    x=theta_vals, y=beta.pdf(theta_vals, 8, 2),
    mode='lines', name='α=8, β=2 (Left-skewed)',
    line=dict(color='red')
))

fig.update_layout(
    title='Beta Distribution Shapes',
    xaxis_title='Conversion Rate (θ)',
    yaxis_title='Probability Density',
    template='plotly_white'
)
fig.show()

# ```

### 2. Sequential Likelihood and Joint History

Since each user interaction is an independent Bernoulli trial, the likelihood of a single observation $y_k \in \{0,1\}$ given the true CTR $\theta$ is:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

The joint likelihood for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

---

### 3. Closed-Form Analytical Updates (Conjugacy)

By Bayes' Theorem, the posterior density at step $k$ is proportional to the product of the likelihood of the new observation and the prior density (which is the posterior from step $k-1$):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting the Bernoulli likelihood and the Beta prior:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \times \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Combine the exponents of $\theta$ and $(1-\theta)$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

Because this resulting expression has the exact algebraic form of a Beta distribution, we have proven **conjugacy**. We do not need to calculate the normalizing integral. The updated parameters are simply:

$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

The Posterior Mean of the latent parameter $\Theta$ at time step $k$ is the expected value of this new Beta distribution:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

---

### 4. Dynamic Shifting Mechanics

Because of Beta-Binomial conjugacy, updating the system is mathematically trivial:
*   **If a user clicks ($y_k = 1$):** The system adds $1$ to $\alpha_{k-1}$. The parameter $\beta_{k-1}$ remains unchanged. This shifts the peak of the probability density to the right.
*   **If a user does not click ($y_k = 0$):** The system adds $1$ to $\beta_{k-1}$. The parameter $\alpha_{k-1}$ remains unchanged. This shifts the peak of the probability density to the left.

**Contrast with Non-Conjugate Setups (e.g., 2PL IRT):**
In a non-conjugate model, multiplying the prior by the likelihood results in an unidentifiable algebraic form. You cannot simply read off new parameters. Instead, the algorithm is forced to perform numerical grid integration across thousands of discrete points at every step just to re-normalize the distribution. Conjugacy bypasses this entirely, allowing continuous, instant updates with zero computational overhead.

---

### 5. Running Point Estimators

Based on the shape parameters $\alpha_k$ and $\beta_k$, the exact closed-form equations for the point estimators at step $k$ are:

**Running Posterior Mean (Expected A Posteriori - EAP):**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

**Running Maximum A Posteriori (MAP - Mode):**
*(Note: The mode is only strictly defined when $\alpha_k > 1$ and $\beta_k > 1$.)*
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$


In [ ]:
# ### 6. Performance Tracking and Convergence Analysis

# ```python
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(42)
n_impressions = 100
theta_true = 0.35

# Initialize prior parameters Beta(1, 1)
alpha_k = 1
beta_k = 1

# Storage for running estimates
map_estimates = []
mean_estimates = []
steps = list(range(1, n_impressions + 1))

# 2. Simulate the Sequential Update Loop
for k in steps:
    # Simulate user interaction (1 = click, 0 = no click)
    y_k = 1 if np.random.rand() < theta_true else 0

    # Analytical Conjugate Update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Calculate Point Estimators
    current_mean = alpha_k / (alpha_k + beta_k)

    # MAP is only strictly defined for alpha > 1 and beta > 1
    if alpha_k > 1 and beta_k > 1:
        current_map = (alpha_k - 1) / (alpha_k + beta_k - 2)
    else:
        # Fallback to mean if MAP is undefined at the boundaries
        current_map = current_mean

    mean_estimates.append(current_mean)
    map_estimates.append(current_map)

# 3. Plotting the Convergence
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=mean_estimates, mode='lines',
    name='Posterior Mean', line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=steps, y=map_estimates, mode='lines',
    name='MAP Estimate', line=dict(color='orange')
))

# True theta reference line
fig.add_hline(y=theta_true, line_dash="dash", line_color="green",
              annotation_text=f"True CTR ({theta_true})")

fig.update_layout(
    title='Convergence of CTR Estimators (100 Impressions)',
    xaxis_title='Impression Step (k)',
    yaxis_title='Estimated Conversion Rate (θ)',
    yaxis=dict(range=[0, 1]),
    template='plotly_white'
)
fig.show()

# ```

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

In [ ]:
# ### 1. Prior Belief Boundaries
#
# The initial prior is modeled as a Beta distribution, $\Theta \sim \text{Beta}(\alpha=8, \beta=1.5)$.
#
# The analytical expected value for the prior stiffness efficiency is given by the mean of the Beta distribution:
#
# $$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$
#
# **Engineering Justification:** This right-skewed distribution is an appropriate prior because a structural component (like an aircraft wing or bridge girder) is assumed to be deployed in a healthy state. The mass of the probability density is concentrated near $1.0$, heavily weighting the assumption that the component is largely intact, while the long left tail acknowledges a small probability of latent manufacturing defects or early-stage degradation.
#
# ```python
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_vals = np.linspace(0.01, 1.0, 500)
prior_pdf = beta.pdf(theta_vals, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theta_vals, y=prior_pdf, mode='lines',
    name='Beta(8, 1.5) Prior', line=dict(color='blue')
))

fig.update_layout(
    title='Initial Prior Belief: Structural Stiffness Efficiency',
    xaxis_title='Stiffness Efficiency Factor (θ)',
    yaxis_title='Probability Density',
    template='plotly_white'
)
fig.show()

# ```

### 2. Structural Likelihood Formulation

The measurement model is multiplicative with log-normal noise:
$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$$

Taking the natural logarithm of both sides yields a normal distribution:
$$\ln(y_k) = \ln(\theta) + \ln(K_{\text{nominal}}) + \epsilon_k$$
$$\ln(y_k) \sim \mathscr{N}(\ln(\theta K_{\text{nominal}}), \sigma^2)$$

By transforming this back, the likelihood of a single continuous sensor measurement $y_k$ given $\theta$ follows the log-normal probability density function:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_k) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

Assuming the sequence of sensor noise is independent, the joint likelihood for the running history vector $\mathbf{y}^{(k)}$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_i) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

---

### 3. Mathematical Formulation of the Non-Conjugate Grid Update

Combining a Beta prior with a Log-Normal likelihood results in a non-conjugate update. The Beta prior contains polynomial terms $\theta^{\alpha-1}(1-\theta)^{\beta-1}$, whereas the likelihood contains an exponential term where $\ln(\theta)$ is squared. Multiplying these together does not yield an algebraic form that matches any standard probability distribution. Consequently, the integral required to normalize the posterior has no closed-form solution.

The recursive relationship up to a proportionality constant is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \exp\left( -\frac{(\ln(y_k) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

*(Note: The leading term $\frac{1}{y_k \sigma \sqrt{2\pi}}$ acts as a constant with respect to $\theta$ and is absorbed into the proportionality).*

---

### 4. Running Point Estimates

Since the posterior is non-standard, point estimates must be evaluated via integration over the bounded domain $(0, 1]$.

**Running Posterior Mean (Bayes Estimator):**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_0^1 \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

**Running Maximum A Posteriori (MAP):**
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

### 5. Algorithmic Grid Approximation and Normalization

To perform this Bayesian update computationally:

1.  **Define the Grid:** Discretize the domain bounds $\theta \in [0.01, 1.0]$ into an array of $M$ evenly spaced points. We start at $0.01$ rather than $0$ to prevent undefined behavior (math domain errors) when computing $\ln(\theta)$.
2.  **Initialize Prior:** Compute the Beta prior probability density evaluated at each grid point.
3.  **Sequential Update:** Upon receiving a new measurement $y_k$:
    *   Evaluate the log-normal likelihood function across the entire $\theta$ grid.
        *   Multiply the likelihood array element-wise by the current posterior array to create an unnormalized density array.
        4.  **Trapezoidal Normalization:**
            *   Use the trapezoidal rule to numerically integrate the unnormalized array over the $\theta$ grid. This yields the marginal likelihood scalar.
                *   Divide the unnormalized array by this scalar to produce the legitimate, normalized probability density for step $k$.
                

In [ ]:
# ### 6. Performance Tracking and Degradation Convergence Analysis

# ```python
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# 1. Simulation Setup
np.random.seed(42)
n_steps = 15
theta_true = 0.68
k_nominal = 50.0
sigma = 0.15

# 2. Grid Initialization
# Start at 0.01 to avoid log(0) domain errors
theta_grid = np.linspace(0.01, 1.0, 1000)
posterior_grid = beta.pdf(theta_grid, 8, 1.5)

# Storage
map_estimates = [theta_grid[np.argmax(posterior_grid)]]
mean_estimates = [np.trapezoid(theta_grid * posterior_grid, theta_grid)]
density_history = {0: posterior_grid.copy()}
steps = [0]

# 3. Simulate and Update
for k in range(1, n_steps + 1):
    # Simulate noisy log-normal measurement
    epsilon_k = np.random.normal(0, sigma)
    y_k = theta_true * k_nominal * np.exp(epsilon_k)

    # Calculate likelihood over the grid
    # Dropping constants 1/(y_k * sigma * sqrt(2pi)) since they normalize out
    likelihood = np.exp(-((np.log(y_k) - np.log(theta_grid * k_nominal))**2) / (2 * sigma**2))

    # Update and normalize via Trapezoidal rule
    unnormalized = likelihood * posterior_grid
    marginal = np.trapezoid(unnormalized, theta_grid)
    posterior_grid = unnormalized / marginal

    # Record estimators
    current_map = theta_grid[np.argmax(posterior_grid)]
    current_mean = np.trapezoid(theta_grid * posterior_grid, theta_grid)

    map_estimates.append(current_map)
    mean_estimates.append(current_mean)
    steps.append(k)

    if k in [1, 2, 5, 10, 15]:
        density_history[k] = posterior_grid.copy()

# 4. Plotting

# Plot 1: Posterior Density Evolution
fig1 = go.Figure()
colors = ['#d3d3d3', '#add8e6', '#87cefa', '#4169e1', '#0000cd', '#000080']
for i, k_hist in enumerate([0, 1, 2, 5, 10, 15]):
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=density_history[k_hist], mode='lines',
        name=f'Step {k_hist}', line=dict(color=colors[i], width=2)
    ))
fig1.add_vline(x=theta_true, line_dash="dash", line_color="red", annotation_text="True Damage (0.68)")
fig1.update_layout(title='Evolution of Posterior Density Distributions', xaxis_title='Stiffness Factor (θ)', yaxis_title='Density', template='plotly_white')
fig1.show()

# Plot 2: Convergence Tracking
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name='Posterior Mean', line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate', line=dict(color='orange')))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="green", annotation_text="True Damage (0.68)")
fig2.update_layout(title='Convergence of Estimators Tracking Structural Damage', xaxis_title='Sensor Measurement Step (k)', yaxis_title='Estimated Stiffness (θ)', template='plotly_white')
fig2.show()

# ```

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

### 1. Deriving the Marginal Density

By the law of total probability, the marginal density of an observation $X_i$ is obtained by marginalizing out the discrete latent variable $C_i$:

$$p(x_i) = \sum_{k=1}^K P(C_i=k) p(X_i=x_i \mid C_i=k)$$

Substituting the given prior probability $P(C_i=k) = \phi_k$ and the conditional Gaussian density $p(X_i=x_i \mid C_i=k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$, we get:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$

**Interpretation:** This density is called a **Gaussian mixture density** because the overall probability distribution of the data is modeled as a weighted linear combination (a "mixture") of $K$ distinct underlying Gaussian distributions. The weights $\phi_k$ dictate the proportion of the dataset originating from each component.

---

### 2. Deriving the Posterior Cluster Probability

Using Bayes' theorem, the posterior probability that observation $x_i$ belongs to cluster $k$ is the joint probability divided by the marginal probability:

$$P(C_i=k \mid X_i=x_i) = \frac{P(X_i=x_i \mid C_i=k)P(C_i=k)}{p(x_i)} = \frac{P(X_i=x_i \mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i \mid C_i=j)P(C_i=j)}$$

Substituting the model definitions:

$$\gamma_{ik} = P(C_i=k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$

**Interpretation:** The responsibility $\gamma_{ik}$ represents the **posterior probability of cluster membership** because it updates our initial prior belief ($\phi_k$) using the observed evidence ($x_i$). It evaluates how likely the point $x_i$ was generated by cluster $k$ relative to all other clusters.

---

### 3. One-Hot Encoding of the Latent Cluster Variable

The variable $Z_{ik}$ is an indicator variable that takes the value $1$ if $C_i=k$ and $0$ otherwise. The expected value of an indicator variable is simply the probability of the event it indicates:

$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(Z_{ik}=1 \mid X_i=x_i) + 0 \cdot P(Z_{ik}=0 \mid X_i=x_i)$$
$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = P(C_i=k \mid X_i=x_i) = \gamma_{ik}$$

Extending this to the vector $Z_i$:

$$\mathbb{E}[Z_i \mid X_i=x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i=x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i=x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

**Conclusion:** The expectation $\mathbb{E}[Z_i \mid X_i=x_i]$ evaluates to the vector of responsibilities. Thus, computing the conditional expectation of the latent variables exactly yields the soft cluster assignments.

---

### 4. From Soft Assignment to Hard Clustering

*   **Soft Clustering:** Represents membership as a probability distribution across all clusters. An observation belongs to cluster 1 with a certain probability, cluster 2 with another, and so on. This captures uncertainty, especially for points near the boundaries between clusters.
*   **Hard Clustering:** Forces a definitive, discrete decision. By taking the `arg max` of the responsibilities ($\widehat{C}_i = \operatorname{arg\,max}_k \gamma_{ik}$), the algorithm assigns the point exclusively to a single cluster, discarding the probabilistic uncertainty.

---

### 5. Conditional Expectation of the Observation

By definition, if $X_i \mid C_i=k$ follows a multivariate normal distribution $\mathscr{N}(\mu_k, \Sigma_k)$, its expected value is its mean parameter:

$$\mathbb{E}[X_i \mid C_i=k] = \int x \mathscr{N}(x \mid \mu_k, \Sigma_k) dx = \mu_k$$

**Interpretation:** The vector $\mu_k$ is the expected location of a data point generated by cluster $k$, effectively making it the spatial center of mass for that cluster.

*   $\mathbb{E}[Z_i \mid X_i=x_i]$ looks "backward" from the data to the latent state, providing the **soft cluster membership** (who generated this point?).
*   $\mathbb{E}[X_i \mid C_i=k]$ looks "forward" from the latent state to the data space, providing the **mean location** of a cluster (where do points from this cluster tend to land?).

---

### 6. The Complete-Data Likelihood

Given the complete data $(x_i, z_i)$ for all $i=1,\dots,n$, the joint likelihood is:

$$p(X, Z) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Taking the natural logarithm, the product over $i$ and $k$ becomes a double summation, and the exponent $z_{ik}$ pulls down as a multiplier:

$$\ell_c = \ln p(X, Z) = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \ln \left( \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right)$$
$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

**Why this is easy to maximize:** Because $z_{ik}$ acts as an on/off switch, the log-likelihood cleanly decouples. For any given cluster $k$, we only sum the logs over the points that explicitly belong to it ($z_{ik}=1$). This turns a complex mixture problem into $K$ separate, simple Gaussian maximum-likelihood estimation problems.

---

### 7. The EM Interpretation

Because $z_{ik}$ is unobserved in practice, the Expectation-Maximization (EM) algorithm replaces it with its conditional expected value given current parameters, $\gamma_{ik}$. Substituting this into the complete-data log-likelihood yields the expected complete-data log-likelihood $Q$:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

**Interpretation of the E-step:** The E-step is a conditional update. Instead of guessing a hard label for $z_{ik}$, we use Bayes' theorem to compute the exact probability that $z_{ik}=1$. We probabilistically distribute the point across the clusters according to how well its location matches each cluster's current mean and variance.

---

### 8. Parameter Updates

By setting the partial derivatives of $Q$ with respect to $\phi_k$, $\mu_k$, and $\Sigma_k$ to zero, we obtain the standard GMM updates.

The term $N_k = \sum_{i=1}^n \gamma_{ik}$ represents the effective number of points assigned to cluster $k$.
The responsibility $\gamma_{ik}$ acts as a **fractional membership weight**. Instead of an observation $x_i$ contributing entirely to one cluster's mean and covariance, it contributes a fraction $\gamma_{ik}$ of its value to cluster $k$. Points closer to a cluster's center will have a $\gamma_{ik}$ near $1$ and heavily influence that cluster, while distant points will have a $\gamma_{ik}$ near $0$ and exert almost no influence.

---

### 9. Interpretation

Gaussian Mixture Model clustering is fundamentally a repeated process of conditional updating. Before observing a specific data point, our belief about its origin is given by the mixture weight $\phi_k$, which serves as the **prior probability** of cluster $k$. When we observe a point $x_i$, we evaluate the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ to measure how highly compatible the spatial location of $x_i$ is with cluster $k$'s distribution. Using Bayes' rule, we combine this compatibility with the prior to compute the responsibility $\gamma_{ik}$, which is the **posterior probability** of cluster $k$ after observing the evidence. This array of posterior probabilities forms the soft assignment vector, $\mathbb{E}[Z_i \mid X_i=x_i]$. During the M-step, the algorithm updates the cluster parameters (means, covariances, and priors) by treating these posterior probabilities as fractional weights. Therefore, Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

### 10. Computational Simulation and Out-of-Sample Validation

```python
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(n_components=self.n_components,
                                   random_state=self.random_state,
                                   max_iter=500)
        
    def preprocess_and_split(self, df, feature_cols):
        # Drop NaNs for the selected features
        df_clean = df[feature_cols].dropna()
        
        # Standardize features
        X_scaled = self.scaler.fit_transform(df_clean)
        
        # Split 80/20
        self.X_train, self.X_test = train_test_split(X_scaled, test_size=0.2, random_state=self.random_state)
        self.feature_names = feature_cols
        print(f"Data split: {self.X_train.shape[0]} train samples, {self.X_test.shape[0]} test samples.")
        
    def fit_evaluate(self):
        # Fit EM algorithm
        self.gmm.fit(self.X_train)
        
        print("\n--- EM Execution ---")
        print(f"Converged: {self.gmm.converged_}")
        print(f"Iterations required: {self.gmm.n_iter_}")
        
        # Out-of-sample performance
        test_log_likelihood = self.gmm.score(self.X_test)
        print(f"\nAverage Log-Likelihood (Test Set): {test_log_likelihood:.4f}")
        
    def plot_density_heatmap(self):
        fig = px.density_contour(
            x=self.X_train[:, 0], y=self.X_train[:, 1],
            marginal_x="histogram", marginal_y="histogram",
            labels={'x': self.feature_names[0], 'y': self.feature_names[1]},
            title="1. 2D Empirical Density Heatmap (Training Data)"
        )
        fig.update_traces(contours_coloring="fill", colorscale="Viridis")
        fig.show()

    def _generate_contour_grid(self, X):
        # Create a mesh grid
        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        grid = np.c_[xx.ravel(), yy.ravel()]
        
        # Get hard assignments for the contour map
        Z = self.gmm.predict(grid)
        Z = Z.reshape(xx.shape)
        return xx, yy, Z

    def plot_training_assignments(self):
        xx, yy, Z = self._generate_contour_grid(self.X_train)
        preds = self.gmm.predict(self.X_train)
        
        fig = go.Figure()
        # Background contour map
        fig.add_trace(go.Contour(x=xx[0], y=yy[:,0], z=Z, colorscale='Pastel', showscale=False, opacity=0.5))
        # Training scatter
        fig.add_trace(go.Scatter(x=self.X_train[:, 0], y=self.X_train[:, 1], mode='markers',
                                 marker=dict(color=preds, colorscale='Darkmint', size=5, line=dict(width=0.5, color='white')),
                                 name='Train Data'))
        fig.update_layout(title="2. Training Assignment Plot with Decision Boundaries",
                          xaxis_title=self.feature_names[0] + " (Scaled)",
                          yaxis_title=self.feature_names[1] + " (Scaled)")
        fig.show()

    def plot_test_assignments(self):
        xx, yy, Z = self._generate_contour_grid(self.X_train) # Use train bounds for consistency
        test_preds = self.gmm.predict(self.X_test)
        
        fig = go.Figure()
        # Background contour map
        fig.add_trace(go.Contour(x=xx[0], y=yy[:,0], z=Z, colorscale='Pastel', showscale=False, opacity=0.5))
        # Test scatter
        fig.add_trace(go.Scatter(x=self.X_test[:, 0], y=self.X_test[:, 1], mode='markers',
                                 marker=dict(color=test_preds, colorscale='Darkmint', size=6, symbol='diamond', line=dict(width=0.5, color='black')),
                                 name='Test Data'))
        fig.update_layout(title="3. Test Assignment Plot (Out-of-Sample Validation)",
                          xaxis_title=self.feature_names[0] + " (Scaled)",
                          yaxis_title=self.feature_names[1] + " (Scaled)")
        fig.show()

# =======================================================
# Execution Block
# =======================================================

if __name__ == "__main__":
    try:
        # Attempt to load Kaggle Dataset
        df = pd.read_csv('CC GENERAL.csv')
        print("Kaggle dataset loaded successfully.")
    except FileNotFoundError:
        print("CC GENERAL.csv not found. Simulating dummy financial data...")
        from sklearn.datasets import make_blobs
        X_dummy, _ = make_blobs(n_samples=5000, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=42)
        df = pd.DataFrame(X_dummy, columns=['PURCHASES', 'CREDIT_LIMIT'])
        
    features = ['PURCHASES', 'CREDIT_LIMIT']
    
    segmenter = GMMFinancialSegmenter(n_components=3)
    segmenter.preprocess_and_split(df, features)
    segmenter.fit_evaluate()
    
    segmenter.plot_density_heatmap()
    segmenter.plot_training_assignments()
    segmenter.plot_test_assignments()
